In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from HHO_kernel import HHO_kernel, HHO_cell

plt.rcParams.update({
    # Make tick labels point inside
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    # Set thicker axis lines (spines)
    'axes.linewidth': 1.5,
})

# Define a prototypical cell

In [ ]:
k = 5 # cell degree
xL = 2 # left boundary
xR = 2.5 # right boundary
cell = HHO_cell(xL,xR,k)

# Check the computation of the monomial basis functions

In [ ]:
# Evaluate basis
xx = np.linspace(xL,xR,50)
basis, gradient = cell.evaluate_basis(xx)

# Plot the results
fig, axs = plt.subplots(1, 2,
                        figsize=(10,5))
for kk in range(k+1):
    label = rf"$\phi_{kk} \sim x^{kk}$"
    axs[0].plot(xx, basis[:,kk], label=label)
    axs[1].plot(xx, gradient[:,kk])
for ax in axs:
    ax.tick_params(direction='in')
    ax.set_xlabel('x')
axs[0].set_title("Basis functions")
axs[1].set_title("Derivative")
axs[0].set_ylabel(r"$\phi(x)$")
axs[1].set_ylabel(r"$\phi'(x)$")
axs[0].legend()
fig.suptitle("Monomial basis");


# Check implementation of the L2-projection

We compute the mass matrix for the case of a low polynomial degree for analytical validation (k=2).

In [ ]:
# Define test example
cell_test_mass = HHO_cell(-1,1,2)
mass = cell_test_mass.mass_cho
lower = cell_test_mass._mass_cho_lower
# Compute the actual mass matrix from the computed cholesky decomposition
# (not to be used in the kernel since matrix multiplication is unstable)
if lower:
    mass = np.tril(mass) @ np.tril(mass).T 
else:
    mass = np.triu(mass).T @ np.triu(mass)
print(mass)

This is exactly the expected result (up to computer precision).
Next we show the projection of a polynomial and a cosine function.

In [ ]:
# Function to compute the L2 projection of a function f and plot the function on the axis ax along with the computed projection
def test_L2_projection(f, ax):
    # Compute projection
    f_proj = cell.evaluate_fun(xx, cell.L2_projection(f))
    # Plot f
    ax.plot(xx,f(xx),'r.',label="f")
    # Compute error in the max norm
    err_max = np.max(np.abs( f(xx) - f_proj ))
    # Plot the projection on the cell
    ax.plot(xx,f_proj,'b-',label=f"L2 projection\nMax error: {err_max:.1e}")
    
fig, axs = plt.subplots(1, 2, 
                        figsize=(12, 4),
                        gridspec_kw={'wspace': 0.8})
f = [lambda x: (x-xL)**5 - (x-xL)*(x-xR),
     lambda x: np.cos(2*np.pi/(xR-xL)*(x-xL))]
for i,ax in enumerate(axs):    
    test_L2_projection(f[i], ax)
    ax.legend(loc='center left',
              bbox_to_anchor=(1, 0.5))
    ax.tick_params(direction='in')
    ax.grid(True, 
            which='major', 
            linestyle='--', 
            linewidth=0.5, 
            color='lightgrey')
axs[0].set_title("5th order polynomial")
axs[1].set_title("Cosine")
fig.suptitle("Example of L2 projections");


As expected, the projection of the polynomial is exact. The projection error of any sufficiently regular function $f$ should satisfy
$$
\lVert f - \Pi^k(f) \rVert_{L^2(\Omega)} \leq C h^{k+1},
$$
where $h$ is the grid spacing and $k$ the polynomial degree. This behaviour is validated below.

In [ ]:
x_left = 0
x_right = 1
N_cells = 4*2**np.arange(0,8)
N_error = 10 # subgrid used in each cell to compute the projection error
f = lambda x : np.cos(4*np.pi*x)
test_degrees = [2, 5, 6]

fig, axs = plt.subplots(1,3, figsize=(12,4))

for (ik, k_test) in enumerate(test_degrees):
    ax = axs[ik]
    errors_L2 = np.zeros(len(N_cells))
    for (i,N) in enumerate(N_cells):
        grid_N = np.linspace(x_left, x_right, N+1)
        solver = HHO_kernel(grid_N, k_test)
        error_L2 = 0
        for c in solver.cells:
            # Approximate the L2 integral of the error of the projection on a fine grid
            grid_error = np.linspace(c.x_left, c.x_right, N_error+1)
            f_proj = c.evaluate_fun(grid_error, c.L2_projection(f))
            error = f_proj - f(grid_error)
            # Add contribution to the L2 error in the current cell
            error_L2 += np.sqrt( 1.0/N_error * np.sum(error**2) )
        errors_L2[i] = error_L2
    ax.plot(1.0/N_cells, errors_L2, '*-', linewidth=0.8, label=r"$\|| f - \Pi^k(f) ||_{L^2(\Omega)}$")
    ax.plot(1.0/N_cells, (10.0/N_cells)**k_test, label=fr"$Ch^{k_test}$")
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('$h$')
    ax.grid(True, 
                which='major', 
                linestyle='--', 
                linewidth=0.5, 
                color='lightgrey')
    ax.legend();


We clearly see that the correct convergence rate is reached until the error is dominated by rounding errors.